In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score

# 1. 데이터 로드
df = pd.read_csv('/Users/leeseungbo/Desktop/학회 데이터(CSV)/5차 전처리 데이터.csv')
target_names = ['사망발생', '중상발생', '경상발생']
X = df.drop(columns=target_names)

# 2. 범주형 변수 처리
for col in X.columns:
    if X[col].dtype == 'object':
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))



# 3. 요청하신 최적 하이퍼파라미터 설정
best_params = {
    'max_depth': 3,
    'n_estimators': 200,
    'learning_rate': 0.1,
    'subsample': 0.7,
    'colsample_bytree': 0.4,
    'gamma': 0.06,
    'reg_alpha': 0.2,
    'random_state': 42,
    'eval_metric': 'logloss'
}

# 4. 개별 타겟 학습 루프
for target in target_names:
    print(f"\n>>> [{target}] 모델 학습 및 평가")
    y = df[target]
    
    # 7:3 분할
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    
    # 클래스 불균형 보정값 계산
    spw = (len(y_train) - sum(y_train)) / sum(y_train)
    
    # 모델 생성 (파라미터 적용)
    model = xgb.XGBClassifier(**best_params, scale_pos_weight=spw)
    
    # 학습
    model.fit(X_train, y_train)
    
    # 평가
    y_prob = model.predict_proba(X_test)[:, 1]
    print(f"ROC AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
    print(classification_report(y_test, model.predict(X_test)))

In [ ]:
# %% [1] 라이브러리 로드 및 한글 폰트 설정
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import platform
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 한글 폰트 설정 (Windows: Malgun Gothic, Mac: AppleGothic)
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False # 마이너스 기호 깨짐 방지

# %% [2] 데이터 로드 및 전처리
# 파일명을 실제 경로에 맞춰 수정해주세요
df = pd.read_csv('/Users/leeseungbo/Desktop/학회 데이터(CSV)/5차 전처리 데이터.csv')

target_names = ['사망발생', '중상발생', '경상발생']
X_raw = df.drop(columns=target_names)

# 범주형 변수 라벨 인코딩
X = X_raw.copy()
le_dict = {}
for col in X.columns:
    if X[col].dtype == 'object':
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        le_dict[col] = le

# %% [3] 모델 및 SHAP 통합 분석 함수 정의
def run_shap_analysis(target_name, spw, mcw):
    print(f"\n{'='*50}")
    print(f" 분석 대상: {target_name}")
    print(f"{'='*50}")
    
    y = df[target_name]
    
    # 7:3 분할
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    
    # 요청하신 최적 파라미터 적용
    model = xgb.XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=3,
        colsample_bytree=0.4,
        gamma=0.06,
        subsample=0.7,
        reg_alpha=0.2,
        scale_pos_weight=spw,
        min_child_weight=mcw,
        random_state=42,
        eval_metric='logloss'
    )
    
    model.fit(X_train, y_train)
    
    # SHAP 값 계산
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    
    # 시각화 (Summary Plot)
    plt.figure(figsize=(10, 6))
    # feature_names를 인코딩 전의 원본 컬럼명으로 지정하여 가독성 확보
    shap.summary_plot(shap_values, X_test, feature_names=X.columns, show=False)
    plt.title(f"[{target_name}] 사고 심각도 결정 요인 (SHAP Summary)", fontsize=15)
    plt.show() # VS Code Interactive 창에 즉시 출력

# %% [4] 사망, 중상, 경상 순차 실행
# (A) 사망발생 분석
run_shap_analysis('사망발생', spw=41.33, mcw=5)

# (B) 중상발생 분석
run_shap_analysis('중상발생', spw=49.95, mcw=7)

# (C) 경상발생 분석
run_shap_analysis('경상발생', spw=14.45, mcw=3)

#### 논문 인용(파라미터) 

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score

# 1. 데이터 로드
# 파일명이 '5차 전처리 데이터.csv'인지 확인해주세요.
df = pd.read_csv('/Users/leeseungbo/Desktop/학회 데이터(CSV)/5차 전처리 데이터.csv')

# 타겟 및 독립변수 설정
target_names = ['사망발생', '중상발생', '경상발생']
X = df.drop(columns=target_names)

# 2. 범주형 데이터 인코딩 (Label Encoding)
le_dict = {}
for col in X.columns:
    if X[col].dtype == 'object':
        X[col] = X[col].fillna('미상')
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        le_dict[col] = le

# 3. 타겟별 개별 설정 (가중치 및 최소 자식 노드 가중치)
target_configs = {
    '사망발생': {'spw': 41.33, 'mcw': 5},
    '중상발생': {'spw': 49.95, 'mcw': 7},
    '경상발생': {'spw': 14.45, 'mcw': 3}
}

# 4. 공통 하이퍼파라미터 (요청하신 값으로 업데이트)
base_params = {
    'n_estimators': 200,      # 요청값
    'learning_rate': 0.1,     # 요청값
    'max_depth': 3,           # 요청값
    'colsample_bytree': 0.4,  # 요청값
    'gamma': 0.06,            # 요청값
    'subsample': 0.7,         # 요청값
    'reg_alpha': 0.2,         # 요청값
    'reg_lambda': 1.0,        # 기본 베이스 유지
    'random_state': 42,
    'eval_metric': 'logloss'
}

# 5. 모델 학습 루프
for target in target_names:
    print(f"\n{'='*40}")
    print(f" 모델 학습: {target}")
    print(f"{'='*40}")
    
    y = df[target]
    
    # 7:3 데이터 분할 (층화 추출 적용)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    
    # 타겟별 특화 파라미터 적용
    config = target_configs[target]
    model_params = base_params.copy()
    model_params['scale_pos_weight'] = config['spw']
    model_params['min_child_weight'] = config['mcw']
    
    # 모델 생성 및 학습
    model = xgb.XGBClassifier(**model_params)
    model.fit(X_train, y_train)
    
    # 결과 예측
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    # 성능 출력
    auc_score = roc_auc_score(y_test, y_prob)
    print(f"\n[{target}] 예측 성능 (ROC AUC): {auc_score:.4f}")
    print(classification_report(y_test, y_pred))

print("\n학습이 완료되었습니다. 이 수치를 논문의 '모델 성능' 섹션에 사용하시면 됩니다.")

In [ ]:
# %% [1] 라이브러리 로드 및 한글 폰트 설정
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import platform
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 한글 폰트 설정 (Windows: Malgun Gothic, Mac: AppleGothic)
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False # 마이너스 기호 깨짐 방지

# %% [2] 데이터 로드 및 전처리
# 파일명을 실제 경로에 맞춰 수정해주세요
df = pd.read_csv('/Users/leeseungbo/Desktop/학회 데이터(CSV)/5차 전처리 데이터.csv')

target_names = ['사망발생', '중상발생', '경상발생']
X_raw = df.drop(columns=target_names)

# 범주형 변수 라벨 인코딩
X = X_raw.copy()
le_dict = {}
for col in X.columns:
    if X[col].dtype == 'object':
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        le_dict[col] = le

# %% [3] 모델 및 SHAP 통합 분석 함수 정의
def run_shap_analysis(target_name, spw, mcw):
    print(f"\n{'='*50}")
    print(f" 분석 대상: {target_name}")
    print(f"{'='*50}")
    
    y = df[target_name]
    
    # 7:3 분할
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    
    # 요청하신 최적 파라미터 적용
    model = xgb.XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=3,
        colsample_bytree=0.4,
        gamma=0.06,
        subsample=0.7,
        reg_alpha=0.2,
        scale_pos_weight=spw,
        min_child_weight=mcw,
        random_state=42,
        eval_metric='logloss'
    )
    
    model.fit(X_train, y_train)
    
    # SHAP 값 계산
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    
    # 시각화 (Summary Plot)
    plt.figure(figsize=(10, 6))
    # feature_names를 인코딩 전의 원본 컬럼명으로 지정하여 가독성 확보
    shap.summary_plot(shap_values, X_test, feature_names=X.columns, show=False)
    plt.title(f"[{target_name}] 사고 심각도 결정 요인 (SHAP Summary)", fontsize=15)
    plt.show() # VS Code Interactive 창에 즉시 출력

# %% [4] 사망, 중상, 경상 순차 실행
# (A) 사망발생 분석
run_shap_analysis('사망발생', spw=41.33, mcw=5)

# (B) 중상발생 분석
run_shap_analysis('중상발생', spw=49.95, mcw=7)

# (C) 경상발생 분석
run_shap_analysis('경상발생', spw=14.45, mcw=3)

#### 2차수정 

In [ ]:
# %% [1] 라이브러리 로드 및 한글 설정
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
import platform

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

# OS별 한글 폰트 설정(그래프는 안 쓰지만, 남겨도 무방)
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False


# %% [2] 데이터 로드 및 One-Hot Encoding
file_path = '/Users/leeseungbo/Desktop/학회 데이터(CSV)/5차 전처리 데이터(1).csv'
df = pd.read_csv(file_path)

target_names = ['사망발생', '중상발생', '경상발생']
X_raw = df.drop(columns=target_names)
X_encoded = pd.get_dummies(X_raw)


# %% [3] 모델 학습 + 성능 출력(그래프/중요도/Top20 출력 없음)
def train_eval_xgb_only_metrics(target_name: str, spw: float, mcw: float, threshold: float = 0.5):
    print("\n" + "="*60)
    print(f"모델 학습: {target_name}")
    print("="*60 + "\n")

    y = df[target_name]

    X_train, X_test, y_train, y_test = train_test_split(
        X_encoded, y, test_size=0.3, random_state=42, stratify=y
    )

    model = xgb.XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=3,
        colsample_bytree=0.4,
        gamma=0.06,
        subsample=0.7,
        reg_alpha=0.2,
        scale_pos_weight=spw,
        min_child_weight=mcw,
        random_state=42,
        eval_metric='logloss',
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    # ROC AUC
    y_proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_proba)

    # classification_report
    y_pred = (y_proba >= threshold).astype(int)

    print(f"[{target_name}] 예측 성능 (ROC AUC): {auc:.4f}")
    print(classification_report(y_test, y_pred, digits=2))

    return model


# %% [4] 실행(성능만 출력)
death_model = train_eval_xgb_only_metrics('사망발생', spw=41.33, mcw=5, threshold=0.5)
serious_model = train_eval_xgb_only_metrics('중상발생', spw=49.95, mcw=7, threshold=0.5)
slight_model = train_eval_xgb_only_metrics('경상발생', spw=14.45, mcw=3, threshold=0.5)


In [ ]:
# %% [1] 라이브러리 로드 및 한글 설정
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import platform

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

# OS별 한글 폰트 설정 (VS Code Interactive Window용)
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False


# %% [2] 데이터 로드 및 One-Hot Encoding
file_path = '/Users/leeseungbo/Desktop/학회 데이터(CSV)/5차 전처리 데이터(1).csv'
df = pd.read_csv(file_path)

target_names = ['사망발생', '중상발생', '경상발생']
X_raw = df.drop(columns=target_names)

# 범주형을 세부 항목별 One-Hot으로 변환 (y축에 졸음/야간 등 개별 표시)
X_encoded = pd.get_dummies(X_raw)


# %% [3] 타깃별 학습 + 성능 출력 + SHAP summary plot (동기화 버전)
def run_xgb_with_shap(
    target_name: str,
    mcw: float,
    threshold: float = 0.5,
    shap_max_display: int = 20,
    shap_sample_size: int = 3000
):
    print("\n" + "="*60)
    print(f"모델 학습: {target_name}")
    print("="*60 + "\n")

    y = df[target_name]

    # 7:3 분할 (층화)
    X_train, X_test, y_train, y_test = train_test_split(
        X_encoded, y, test_size=0.3, random_state=42, stratify=y
    )

    # 클래스 불균형: train 기준 동적 계산 (당신 SHAP 코드와 동일 로직)
    pos = (y_train == 1).sum()
    neg = (y_train == 0).sum()
    if pos == 0:
        raise ValueError(f"[{target_name}] 학습 데이터에 양성(1)이 0개입니다. 분할/데이터를 확인하세요.")
    spw = neg / pos

    # 레퍼런스 논문에서 가져온 하이퍼파라미터 "그대로" + spw/mcw만 타깃별 적용
    model = xgb.XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=3,
        colsample_bytree=0.4,
        gamma=0.06,
        subsample=0.7,
        reg_alpha=0.2,
        scale_pos_weight=spw,
        min_child_weight=mcw,
        random_state=42,
        eval_metric='logloss',
        n_jobs=-1
    )


    model.fit(X_train, y_train)

    # -------------------
    # (A) 성능 출력 (스크린샷 형태)
    # -------------------
    y_proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_proba)
    y_pred = (y_proba >= threshold).astype(int)

    print(f"[{target_name}] 예측 성능 (ROC AUC): {auc:.4f}")
    print(classification_report(y_test, y_pred, digits=2))

    # -------------------
    # (B) SHAP summary plot
    # -------------------
    # SHAP은 데이터가 크면 느리므로 샘플링(기본 3000) — 결과 확인용
    # 필요시 shap_sample_size를 None 또는 더 크게 조절 가능
    if shap_sample_size is not None and len(X_test) > shap_sample_size:
        X_shap = X_test.sample(shap_sample_size, random_state=42)
    else:
        X_shap = X_test

    # TreeExplainer
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_shap)

    plt.figure(figsize=(12, 10))
    shap.summary_plot(shap_values, X_shap, max_display=shap_max_display, show=False)
    plt.title(f"[{target_name}] 세부 위험 요인 기여도 분석 (SHAP Summary)", fontsize=16)
    plt.show()

    return model


# %% [4] 실행 (타깃별 mcw만 다르게 적용)
# 사망/중상: 희귀이므로 mcw를 상대적으로 크게(7), 경상은 3
death_model   = run_xgb_with_shap('사망발생', mcw=7, threshold=0.5, shap_max_display=20, shap_sample_size=3000)
serious_model = run_xgb_with_shap('중상발생', mcw=7, threshold=0.5, shap_max_display=20, shap_sample_size=3000)
slight_model  = run_xgb_with_shap('경상발생', mcw=3, threshold=0.5, shap_max_display=20, shap_sample_size=3000)


####